In [ ]:
!pip install -q -U transformers sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM

print("Phiên bản PyTorch:", torch.__version__)
print("Có GPU không?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tên GPU:", torch.cuda.get_device_name(0))

Phiên bản PyTorch: 2.11.0+cu130
Có GPU không? True
Tên GPU: Tesla T4


In [ ]:
"""
Demo PhoBERT (VinAI) - pip install torch transformers
Lưu ý: PhoBERT yêu cầu văn bản ĐÃ TÁCH TỪ (word segmentation),
các âm tiết của một từ ghép nối bằng dấu "_" (vd: "sinh_viên").
Thực tế nên dùng VnCoreNLP / py_vncorenlp để tách từ (cần Java);
ở đây dùng câu đã tách sẵn để chạy minh họa nhanh.
"""
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM

MODEL_NAME = "vinai/phobert-base-v2"   # hoặc "vinai/phobert-base", "vinai/phobert-large"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

# Câu đã tách từ
sentence = "Chúng_tôi là những nghiên_cứu_viên đang tìm_hiểu về xử_lý ngôn_ngữ tự_nhiên ."

# 1) Tokenize + trích xuất embedding
inputs = tokenizer(sentence, return_tensors="pt").to(device)
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

with torch.no_grad():
    outputs = model(**inputs)

last_hidden = outputs.last_hidden_state            # (1, seq_len, 768)
cls_vec = last_hidden[:, 0, :]                     # vector <s>
mean_vec = last_hidden.mean(dim=1)                 # mean pooling
print("Hidden states shape:", tuple(last_hidden.shape))
print("CLS vector (5 giá trị đầu):", cls_vec[0, :5].tolist())

# 2) Fill-mask
mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()
masked = "Hà_Nội là thủ_đô của <mask> ."
inp = tokenizer(masked, return_tensors="pt").to(device)
with torch.no_grad():
    logits = mlm(**inp).logits
mask_idx = (inp["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
top5 = torch.topk(logits[0, mask_idx], 5, dim=-1).indices[0]
print("\nFill-mask:", masked)
for i, tid in enumerate(top5, 1):
    print(f"  {i}. {tokenizer.decode([tid])}")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (3 outputs left out)

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokens: ['<s>', 'Chúng_tôi', 'là', 'những', 'nghiên_cứu_viên', 'đang', 'tìm_hiểu', 'về', 'xử_lý', 'ngôn_ngữ', 'tự_nhiên', '.', '</s>']
Hidden states shape: (1, 13, 768)
CLS vector (5 giá trị đầu): [0.1868048757314682, 0.6869877576828003, -0.4620154798030853, -0.6148040890693665, -0.31377264857292175]


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


Fill-mask: Hà_Nội là thủ_đô của <mask> .
  1. Việt_Nam
  2. chúng_ta
  3. Cộng_hoà_Xã_hội_Chủ_nghĩa_Việt_Nam
  4. đất_nước
  5. tổ_quốc


In [ ]:
"""
Demo viBERT (FPT AI) - pip install torch transformers
viBERT là BERT đa ngữ được huấn luyện tiếp / điều chỉnh từ vựng cho tiếng Việt.
KHÔNG bắt buộc tách từ như PhoBERT (dùng WordPiece trên văn bản thô).
"""
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM

MODEL_NAME = "FPTAI/vibert-base-cased"
# Biến thể khác để so sánh: "NlpHUST/vibert4news-base-cased"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

sentence = "Chúng tôi là những nghiên cứu viên đang tìm hiểu về xử lý ngôn ngữ tự nhiên."

# 1) Tokenize + embedding
inputs = tokenizer(sentence, return_tensors="pt").to(device)
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

with torch.no_grad():
    outputs = model(**inputs)

last_hidden = outputs.last_hidden_state
cls_vec = last_hidden[:, 0, :]                     # vector [CLS]
mean_vec = last_hidden.mean(dim=1)
print("Hidden states shape:", tuple(last_hidden.shape))
print("CLS vector (5 giá trị đầu):", cls_vec[0, :5].tolist())

# 2) Fill-mask
try:
    mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()
    masked = f"Hà Nội là thủ đô của {tokenizer.mask_token} ."
    inp = tokenizer(masked, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = mlm(**inp).logits
    mask_idx = (inp["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
    top5 = torch.topk(logits[0, mask_idx], 5, dim=-1).indices[0]
    print("\nFill-mask:", masked)
    for i, tid in enumerate(top5, 1):
        print(f"  {i}. {tokenizer.decode([tid])}")
except Exception as e:
    print("\n(Checkpoint này có thể không kèm MLM head, bỏ qua fill-mask):", e)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/255k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  581MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: FPTAI/vibert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (2 outputs left out)

Tokens: ['[CLS]', 'chung', 'toi', 'la', 'n', '##hung', 'nghi', '##en', 'cu', '##u', 'vien', 'đang', 'tim', 'hie', '##u', 've', 'x', '##u', 'ly', 'ng', '##on', 'ng', '##u', 'tu', 'n', '##hien', '.', '[SEP]']
Hidden states shape: (1, 28, 768)
CLS vector (5 giá trị đầu): [0.05722673609852791, 0.6866599321365356, 0.00913949590176344, -0.27543628215789795, 0.09002338349819183]


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] BertForMaskedLM LOAD REPORT from: FPTAI/vibert-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Fill-mask: Hà Nội là thủ đô của [MASK] .
  1. dan
  2. tien
  3. ban
  4. chung
  5. hoc


In [ ]:
"""
Demo XLM-RoBERTa (Meta AI) - pip install torch transformers sentencepiece
Mô hình đa ngữ (100 ngôn ngữ, gồm tiếng Việt), dùng SentencePiece,
KHÔNG cần tách từ trước.
"""
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM

MODEL_NAME = "xlm-roberta-base"        # hoặc "xlm-roberta-large"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

sentence = "Chúng tôi là những nghiên cứu viên đang tìm hiểu về xử lý ngôn ngữ tự nhiên."

# 1) Tokenize + embedding
inputs = tokenizer(sentence, return_tensors="pt").to(device)
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

with torch.no_grad():
    outputs = model(**inputs)

last_hidden = outputs.last_hidden_state
cls_vec = last_hidden[:, 0, :]                     # vector <s>
mean_vec = last_hidden.mean(dim=1)
print("Hidden states shape:", tuple(last_hidden.shape))
print("CLS vector (5 giá trị đầu):", cls_vec[0, :5].tolist())

# 2) Fill-mask
mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()
masked = "Hà Nội là thủ đô của <mask> ."
inp = tokenizer(masked, return_tensors="pt").to(device)
with torch.no_grad():
    logits = mlm(**inp).logits
mask_idx = (inp["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
top5 = torch.topk(logits[0, mask_idx], 5, dim=-1).indices[0]
print("\nFill-mask:", masked)
for i, tid in enumerate(top5, 1):
    print(f"  {i}. {tokenizer.decode([tid])}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

Tokens: ['<s>', '▁Chúng', '▁tôi', '▁là', '▁những', '▁nghiên', '▁cứu', '▁viên', '▁đang', '▁tìm', '▁hiểu', '▁về', '▁xử', '▁lý', '▁ngôn', '▁ngữ', '▁tự', '▁nhiên', '.', '</s>']
Hidden states shape: (1, 20, 768)
CLS vector (5 giá trị đầu): [0.049364395439624786, 0.10427343100309372, 0.06629175692796707, -0.03556302562355995, 0.06103513389825821]


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Fill-mask: Hà Nội là thủ đô của <mask> .
  1. Pháp
  2. tôi
  3. Indonesia
  4. bạn
  5. Đức
